In [4]:
# @app.route('/set_destination', methods=['POST'])
# def set_destination():
#     """
#     유니티에서 목표지점을 설정하면 현재위치를 기반으로 최단 경로를 계산하여 이동 명령을 생성하는 API
#     """
#     global move_command, current_position, target_path
#     data = request.json
#     destination = data.get("destination")
#     #obstacles = data.get("obstacles",[])
#     print(destination)

#     #현재 위치가 저장되지 않았으면에러 반환
#     if current_position is None:
#         return "현재 위치 저장 않됨"

In [12]:
# plask 라이브러리 이용하여 서버 구동해보자

from flask import Flask, request, jsonify

app = Flask(__name__) # 플라스크 객체 선언

move_command = "W W W D D D D D W W W A A A A A A W W W S S S"

user = [] # user라는 배열 전역변수로 할당

@app.route('/', methods = ['GET'])
# 라우트 멍령어: 함수 실행하기 위해 어떤 URL 사용할 것인가 지정, '/' -> 루트이다
# 헬로월드 켓 매소드로 받아보기
def home():
    return 'hello world'

@app.route('/info', methods = ['GET'])    
def info():
    title = "안녕하세요"
    return f"""
    <html>
    <head>
        <title>INFO</title>
    </head>
    <body>
        <h1>{title} 플라스크 서버정보</h1>
        <a href = '/sec'>두번째 페이지</a>
    </body>
    </html>
    """

# @app.route('/sec', methods = ['GET'])
# def sec():
#     name = request.args.get('name')
#     age = request.args.get('age')
#     gender = request.args.get("gender")
    
#     return f"""
#     <html>
#     <head>
#         <title> INFO - sec </title>
#     </head>
#     <body>
#         <h1> 두번째 페이지</h1>
#         <h3>이름: {name} </h3>
#         <h3>나이: {age} </h3>
#         <h3>성별: {gender} </h3>
#     </body>
#     </html>
#     """

  
  
@app.route('/secPost', methods = ['POST'])
def secPost():
    name = request.form.get('name')
    age = request.form.get('age')
    gender = request.form.get("gender")
    
    return f"""
    <html>
    <head>
        <title> INFO - sec </title>
        <script>
            alert('{name}님 안녕하세요');
        </script>
    </head>
    <body>
        <h1> 두번째 페이지</h1>
        <h3>이름: {name} </h3>
        <h3>나이: {age} </h3>
        <h3>성별: {gender} </h3>
    </body>
    </html>
    """

@app.route('/sec', methods = ['GET'])
def sec():
    name = request.args.get('name')
    age = request.args.get('age')
    gender = request.args.get("gender")
    
    return f"""
    <html>
    <head>
        <title> INFO - sec </title>
    </head>
    <body>
        <h1> 두번째 페이지</h1>
        <form method = 'post' action = "/secPost">
        
            <h3>이름: <input type = 'text' name = 'name' 'value = '{name}'> </h3>
            <h3>나이: <input type = 'text' name = 'age' 'value = '{name}'> </h3>
            <h3>성별: <input type = 'text' name = 'gender' 'value = '{name}'> </h3>
            <button type = 'submit'> 전송</button>
        </form>
        
    </body>
    </html>
    """

@app.route('/result', methods = ['POST'])
def result():
    return jsonify({'result':'success'})

@app.route('/userinfo', methods = ['POST'])
def userinfo():
    name = request.form.get("name")
    age = request.form.get("age")
    gender = request.form.get("gender")
    
    return jsonify({'name':name,
                   'age':age,
                   'gender':gender})


@app.route('/userAdd', methods = ['POST'])
def userAdd():
    global user # 전역변수 사용하기
    name = request.form.get("name")
    age = request.form.get("age")
    gender = request.form.get("gender")
    
    userData = {'name':name,
                   'age':age,
                   'gender':gender}
    user.append(userData)
    
    return jsonify(userData)

@app.route('/userAll', methods = ['GET'])
def userAll():
    global user # 전역변수 사용하기
    return jsonify(user)

@app.route('/userGet', methods = ['GET'])
def userGet():
    global user # 전역변수 사용하기
    idx = int(request.args.get('idx'))
    if len(user) <= idx:
        return jsonify({'status':'error', 'message':'Out of Index'})
    return jsonify(user[idx])

@app.route('/detect', methods=['POST'])
def detect():
    image = request.files['image']
    image_path = 'temp_image.jpg'
    image.save(image_path)

    model = YOLO('yolov8n.pt')
    result = model(image_path)
    detections = result[0].boxes.data.cpu().numpy()
    print(detections)

    result_json = []
    classes = [
    'person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 
    'truck', 'boat', 'traffic light', 'fire hydrant', 'stop sign', 'parking meter',
    'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow', 'elephant', 'bear', 
    'zebra', 'giraffe', 'backpack', 'umbrella', 'handbag', 'tie', 'suitcase', 
    'frisbee', 'skis', 'snowboard', 'sports ball', 'kite', 'baseball bat', 
    'baseball glove', 'skateboard', 'surfboard', 'tennis racket', 'bottle', 
    'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple', 
    'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 
    'cake', 'chair', 'couch', 'potted plant', 'bed', 'dining table', 'toilet', 
    'TV', 'laptop', 'mouse', 'remote', 'keyboard', 'cell phone', 'microwave', 
    'oven', 'toaster', 'sink', 'refrigerator', 'book', 'clock', 'vase', 
    'scissors', 'teddy bear', 'hair drier', 'toothbrush'
    ]
    
    #for 문은 시퀀스, box의 0의 값이 coord에 종속되고, 플롯형이 됨, 이후 인덱스 0이 인덱스 box[:4]까지 반복
    for box in detections:
        result_json.append({
            'className': classes[int(box[5])],
            'bbox': [float(coord) for coord in box[:4]],
            'confidence': int(box[4])
        })
    
    return jsonify(result_json)


@app.route('/set_destination', methods=['POST'])
def set_destination():
    """
    유니티에서 목표지점을 설정하면 현재위치를 기반으로 최단 경로를 계산하여 이동 명령을 생성하는 API
    """
    global move_command, current_position, target_path
    
    data = request.json
    destination = data.get("destination")
    #obstacles = data.get("obstacles",[])
    print("destination: ", destination)

    try:
        return jsonify({"status":"OK", "MESSAGE":"SUCCESS"})
    except Exception as e:
        return jsonify({"status":"ERROR", "MESSAGE":str(e)}), 400

@app.route('/update_position', methods=["POST"])
def update_position():
    """
    유니티에서 현재 위치를 주기적으로 업데이트 하는 API
    """
    global current_position
    
    data = request.json
    current_position = data.get("current_position")
    print("Current position updated to:", current_position)

    try:
        return jsonify({"status":"OK", "MESSAGE":"Current position updated successfully"})
    except Exception as e:
        return jsonify({"status":"ERROR", "MESSAGE":str(e)}), 400


@app.route('/get_move', methods = ['GET'])
def get_move():
    """
    유니티에서 주기적으로 호출명령하여 이동명령을 가져오는 API
    현재 위치와 목표경로를 비교하여 오차를 수정
    """
    global move_command

    # move_command = "W W W D D D D D W W W A A A A A A W W W S S S"
    # 본 커맨드에선 마지막에 후진 명령을 했지만 먹지 않는다 --> 이는 명령 사이에 공백을 두어 
    # 전진 명령에 대한 브레이크가 걸리게 한 후 명령을 하면 해결가능성이 있다.
    
    if len(move_command) > 0:
        command = move_command[0]
        move_command = move_command[1:] #앞글자 제거
        return jsonify({"move":command})
    else:
        return jsonify({"move":"STOP"})



# 서버 구동을 위한 메인 함수 필요
if __name__ == '__main__':
    app.run(host = '0.0.0.0', port = 5001) # app을 실행해라, 호스트는 0.0.0.0 -> 외부에서도 접근 가능?, 포트는 5001번으로

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5001
 * Running on http://192.168.0.217:5001
Press CTRL+C to quit
127.0.0.1 - - [21/Feb/2025 15:37:51] "POST /set_destination HTTP/1.1" 200 -
127.0.0.1 - - [21/Feb/2025 15:37:51] "POST /update_position HTTP/1.1" 200 -


Current position updated to:destination:  73.18321,9.167738,101.2927
 None


127.0.0.1 - - [21/Feb/2025 15:37:52] "POST /set_destination HTTP/1.1" 200 -
127.0.0.1 - - [21/Feb/2025 15:37:52] "POST /update_position HTTP/1.1" 200 -


destination:  109.8365,14.32366,153.9882
Current position updated to: None


...

Current position updated to: None


127.0.0.1 - - [21/Feb/2025 15:38:28] "POST /update_position HTTP/1.1" 200 -
127.0.0.1 - - [21/Feb/2025 15:38:28] "GET /get_move HTTP/1.1" 200 -


Current position updated to: None


127.0.0.1 - - [21/Feb/2025 15:38:28] "GET /get_move HTTP/1.1" 200 -
127.0.0.1 - - [21/Feb/2025 15:38:28] "POST /update_position HTTP/1.1" 200 -


Current position updated to: None
